# Nova AI — نموذج توليد الصور الخاص بنا (Stable Diffusion، مملوك بالكامل)

**منفصل تماماً عن دفتر `merge_and_finetune.ipynb`** (فهم النص والصور) —
فهم الصور وتوليدها بنيتان مختلفتان جذرياً في الشبكات العصبية، فلا يمكن
لأي قدر من تدريب Qwen2.5-VL أن يكسبه القدرة على توليد صور. هذا نموذج
**ثانٍ ومستقل بالكامل**، لكنه بنفس مبدأ الملكية بالضبط: أوزان مفتوحة
المصدر بالكامل، نُنزّلها، نملك نسختنا الخاصة على مستودعنا، ونستضيفها
مجاناً — وليس استدعاء API مستأجَراً من أي شركة.

**لا حاجة لتشغيل هذا الدفتر أسبوعياً** كدفتر النص/الرؤية — نموذج
Stable Diffusion المُدرَّب مسبقاً يعمل بجودة عالية فور تنزيله دون أي
تدريب إضافي، ولا يوجد مصدر بيانات "يكبر تلقائياً" لتوليد الصور كما
يحدث مع محادثات نوفا النصية. شغّله **مرة واحدة** لتفعيل الميزة، ولاحقاً
مرة أو مرتين فقط إن أردت لاحقاً تخصيص أسلوب بصري مميز لنوفا عبر تدريب
LoRA خفيف (خطوة مستقبلية اختيارية، غير مبنية في هذا الدفتر بعد).

## الإعداد لمرة واحدة فقط

**1) استورد الدفتر** بنفس طريقة الدفتر الآخر تماماً: Kaggle → Create →
New Notebook → File → Import Notebook → GitHub → الصق رابط هذا الملف.
فعّل **GPU T4** من Notebook options.

**2) الأسرار (Secrets):** فقط `HF_TOKEN` و`HF_USERNAME` (نفس القيمتين
المستخدمتين في دفتر النص/الرؤية — إن كان هذا الدفتر في نفس حساب
Kaggle، الأسرار مشتركة تلقائياً ولا حاجة لإضافتها مجدداً).

**3) شغّل الخلايا بالترتيب من الأعلى للأسفل يدوياً هذه المرة** — لا
حاجة لجدولة أسبوعية (Schedule) لهذا الدفتر تحديداً، فقط شغّله عند
الحاجة.

**بعد نجاح الرفع (آخر خلية):** ضع اسم المستودع الذي تطبعه في
`HF_IMAGE_MODEL_ID` على Render (Environment Variables) ثم Manual
Deploy. عندها يعمل أمر `/صورة` في بوت نوفا فوراً.

In [ ]:
# الخلية 1 — تثبيت الأدوات
!pip install -q diffusers accelerate transformers safetensors huggingface_hub

In [ ]:
# الخلية 2 — تسجيل الدخول إلى Hugging Face (بلا أي تدخل يدوي)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("تم تسجيل الدخول إلى Hugging Face بنجاح")

In [ ]:
# الخلية 3 — الأساس المفتوح، ومستودعنا الخاص
#
# Stable Diffusion 2.1 (إصدار "base"، أخف من SDXL ويناسب GPU T4 المجاني
# على Kaggle بذاكرة كافية للتوليد المباشر بلا تعقيد إضافي). أوزان
# مفتوحة بالكامل من Stability AI نفسها (ترخيص CreativeML OpenRAIL —
# يسمح بالاستخدام والتعديل والتوزيع). كتالوج النماذج المفتوحة يتغيّر
# بمرور الوقت مثل Groq/Gemini تماماً — تحقق من
# huggingface.co/stabilityai قبل افتراض أن هذا الاسم لا يزال الأفضل.
from kaggle_secrets import UserSecretsClient

BASE_IMAGE_MODEL_ID = "stabilityai/stable-diffusion-2-1-base"
HF_USERNAME = UserSecretsClient().get_secret("HF_USERNAME")
REPO_ID = f"{HF_USERNAME}/nova-image-gen"
print("الأساس:", BASE_IMAGE_MODEL_ID)
print("سيُرفَع نموذجنا إلى:", REPO_ID)

In [ ]:
# الخلية 4 — تحميل النموذج والتحقق من عمله فعلياً بتوليد صورة تجريبية
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(BASE_IMAGE_MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

test_image = pipe("a friendly cartoon robot mascot, simple flat design", num_inference_steps=25).images[0]
test_image.save("test_output.png")
print("تم توليد صورة تجريبية بنجاح — تحقق من test_output.png في ملفات الجلسة (Output) للتأكد بصرياً.")

In [ ]:
# الخلية 5 — رفع النموذج إلى مستودعنا الخاص على Hugging Face Hub
#
# هذه هي نفس أوزان Stability AI الأصلية، منسوخة بالكامل إلى مستودعنا
# نحن — نملك نسختنا الآن بشكل كامل ومستقل عن أي تغيير مستقبلي على
# المستودع الأصلي، وجاهزة لاستضافة HF Inference المجانية بنفس طريقة
# نموذج النص/الرؤية بالضبط.
pipe.save_pretrained("./nova-image-gen-local")

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-image-gen-local", repo_id=REPO_ID)
print(f"تم الرفع: https://huggingface.co/{REPO_ID}")
print("ضع هذا في HF_IMAGE_MODEL_ID داخل ai-system/.env أو Render:", REPO_ID)